<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dinosaur Training Notebook

Train a dinosaur to walk, run, and perform species-specific behaviors using reinforcement learning.

**Supported Species:**
- **Velociraptor** — Bipedal predator with sickle-claw strike
- **T-Rex** — Large bipedal predator with bite attack
- **Brachiosaurus** — Quadrupedal sauropod with food reaching

**Training Stages (all species):**
1. **Balance** — Learn to stand without falling
2. **Locomotion** — Walk and run forward
3. **Species behavior** — Strike / Bite / Food reach

**Supported Algorithms:** PPO, SAC

Change the `SPECIES` variable in the configuration cell below to select which dinosaur to train.
All configs are loaded from the TOML files in `configs/<species>/`.

## 1. Setup & Installation

In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

# Library imports — shared training infrastructure
from environments.shared.train_base import (
    SpeciesConfig,
    create_vec_env,
    evaluate,
    linear_schedule,
    train,
)
from environments.shared.config import load_all_stages, save_stage_config
from environments.shared.curriculum import (
    RewardRampCallback,
    StageWarmupCallback,
    load_vecnorm_stats,
)

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Change `SPECIES` below to train a different dinosaur. All other settings apply to every species.

In [ ]:
# ===== SPECIES SELECTION =====
# Change this to: "velociraptor", "trex", or "brachiosaurus"
SPECIES = "velociraptor"

# ===== Training settings =====
ALGORITHM = "ppo"      # "ppo" or "sac"
N_ENVS = 4             # Number of parallel environments
SEED = 42              # Random seed for reproducibility
VERBOSE = 1            # 0=quiet, 1=progress bar, 2=debug
QUICK_TEST = False     # True = tiny run to verify setup works
USE_GOOGLE_DRIVE = True  # Mount Google Drive to save results

print(f"Species: {SPECIES}")
print(f"Algorithm: {ALGORITHM.upper()}")
print(f"Quick test mode: {QUICK_TEST}")

In [ ]:
# ============================================================
# Storage Configuration
# ============================================================
# When USE_GOOGLE_DRIVE is True and running in Colab, logs and
# models are saved to Google Drive so they persist across sessions.
# Otherwise, everything is saved to the local filesystem.

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    LOG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")
else:
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")

# Create a single run directory for all stages, organised by species then datetime
RUN_DIR = LOG_BASE / SPECIES / f"{ALGORITHM.lower()}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {RUN_DIR}")

## 3. Explore the Environment

In [ ]:
# Resolve species environment class and build SpeciesConfig
import importlib

_SPECIES_MAP = {
    "velociraptor": {
        "module": "environments.velociraptor.envs.raptor_env",
        "class": "RaptorEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=strike",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["strike_success", "bite_success"],
    },
    "trex": {
        "module": "environments.trex.envs.trex_env",
        "class": "TRexEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=bite",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["bite_success", "strike_success"],
    },
    "brachiosaurus": {
        "module": "environments.brachiosaurus.envs.brachio_env",
        "class": "BrachioEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=food_reach",
        "height_label": "Torso height",
        "stage3_section_label": "Food Reaching",
        "success_keys": ["food_reached"],
    },
}

assert SPECIES in _SPECIES_MAP, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_MAP.keys())}"
_info = _SPECIES_MAP[SPECIES]
_mod = importlib.import_module(_info["module"])
EnvClass = getattr(_mod, _info["class"])

# Build SpeciesConfig (same structure used by the CLI train scripts)
SPECIES_CFG = SpeciesConfig(
    species=SPECIES,
    env_class=EnvClass,
    stage_descriptions=_info["stage_descriptions"],
    height_label=_info["height_label"],
    stage3_section_label=_info["stage3_section_label"],
    success_keys=_info["success_keys"],
)

# Load all stage configs from TOML files
STAGE_CONFIGS = load_all_stages(SPECIES)

env = EnvClass()
print(f"Environment: {EnvClass.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
print(f"Number of actuators: {env.model.nu}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} - {cfg['description']}")
env.close()

In [ ]:
# Run random episodes to establish a baseline
env = EnvClass()
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    total_reward = 0
    step = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        if terminated or truncated:
            break
    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

print("\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
env.close()

## 4. Training Infrastructure

In [ ]:
import time
from collections import Counter

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.utils import get_schedule_fn

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}

try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Videos will be skipped. Install with: pip install mediapy")

# Accumulates (stage_num, stage_dir) tuples as each stage completes,
# so training curves can be plotted incrementally after every stage.
completed_stages = []


# ============================================================
# Diagnostics Callback - logs reward components, obs/action
# stats, VecNormalize state, termination reasons, and plateau
# warnings to TensorBoard
# ============================================================
class DiagnosticsCallback(BaseCallback):
    """Logs per-component reward breakdowns and training diagnostics to TensorBoard.

    Tracked metrics (under ``diagnostics/`` in TensorBoard):
      - Per-component rewards: reward_forward, reward_alive, reward_energy, etc.
      - Environment state: forward_vel, prey_distance, pelvis_height, tilt_angle
      - Observation statistics: mean, std, max absolute value
      - Action statistics: mean, std
      - VecNormalize running variance for observations and returns
      - Termination reason breakdown (fraction per reason)
      - Reward plateau detection with console warnings

    When *log_dir* is provided, per-rollout averages of INFO_KEYS are also
    saved to ``diagnostics.npz`` so they can be plotted alongside the
    evaluation curves produced by :func:`plot_training_curves`.
    """

    REWARD_KEYS = [
        "reward_forward",
        "reward_alive",
        "reward_energy",
        "reward_tail",
        "reward_posture",
        "reward_nosedive",
        "reward_smoothness",
        "reward_strike",
        "reward_approach",
        "reward_gait",
        "reward_heading",
        "reward_lateral",
    ]
    INFO_KEYS = [
        "forward_vel",
        "prey_distance",
        "pelvis_height",
        "tilt_angle",
        "tail_instability",
        "contact_asymmetry",
        "action_delta",
        "heading_alignment",
        "lateral_vel",
        "forward_z",
        "approach_delta",
        "strike_success",
        "r_foot_contact",
        "l_foot_contact",
    ]

    def __init__(self, plateau_window=10, plateau_threshold=1.0, log_dir=None, verbose=0):
        super().__init__(verbose)
        self.plateau_window = plateau_window
        self.plateau_threshold = plateau_threshold
        self._log_dir = Path(log_dir) if log_dir is not None else None
        self._step_infos = {k: [] for k in self.REWARD_KEYS + self.INFO_KEYS}
        self._rollout_ep_rewards = []
        self._rollout_terminations = Counter()
        # Per-rollout history for npz persistence
        self._history_timesteps = []
        self._history = {k: [] for k in self.INFO_KEYS}
        self._history_rewards = {k: [] for k in self.REWARD_KEYS}
        self._history_terminations = {}  # reason -> [fraction per rollout]
        self._history_term_timesteps = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            for key in self.REWARD_KEYS + self.INFO_KEYS:
                if key in info:
                    self._step_infos[key].append(float(info[key]))
            # Track termination reasons from completed episodes
            if "termination_reason" in info:
                self._rollout_terminations[info["termination_reason"]] += 1
        return True

    def _on_rollout_end(self) -> None:
        # Per-component reward breakdown
        for key, values in self._step_infos.items():
            if values:
                self.logger.record(f"diagnostics/{key}", np.mean(values))

        # Persist rollout averages for INFO_KEYS and REWARD_KEYS
        has_info = any(self._step_infos[k] for k in self.INFO_KEYS)
        if has_info:
            self._history_timesteps.append(self.num_timesteps)
            for key in self.INFO_KEYS:
                vals = self._step_infos[key]
                self._history[key].append(float(np.mean(vals)) if vals else float("nan"))
            for key in self.REWARD_KEYS:
                vals = self._step_infos[key]
                self._history_rewards[key].append(float(np.mean(vals)) if vals else float("nan"))
            self._save_diagnostics()

        self._step_infos = {k: [] for k in self.REWARD_KEYS + self.INFO_KEYS}

        # Termination reason breakdown (fraction of terminated episodes)
        total_terminations = sum(self._rollout_terminations.values())
        if total_terminations > 0:
            self._history_term_timesteps.append(self.num_timesteps)
            for reason, count in self._rollout_terminations.items():
                frac = count / total_terminations
                self.logger.record(f"terminations/{reason}", frac)
                self._history_terminations.setdefault(reason, []).append(frac)
            self.logger.record("terminations/total_count", total_terminations)
            self._save_diagnostics()
        self._rollout_terminations.clear()

        # Observation statistics from rollout buffer
        if hasattr(self.model, "rollout_buffer") and self.model.rollout_buffer.observations is not None:
            obs = self.model.rollout_buffer.observations
            self.logger.record("diagnostics/obs_mean", float(np.mean(obs)))
            self.logger.record("diagnostics/obs_std", float(np.std(obs)))
            self.logger.record("diagnostics/obs_max_abs", float(np.max(np.abs(obs))))

        # Action statistics from rollout buffer
        if hasattr(self.model, "rollout_buffer") and self.model.rollout_buffer.actions is not None:
            acts = self.model.rollout_buffer.actions
            self.logger.record("diagnostics/action_mean", float(np.mean(acts)))
            self.logger.record("diagnostics/action_std", float(np.std(acts)))

        # VecNormalize running statistics
        env = self.training_env
        if hasattr(env, "obs_rms"):
            self.logger.record("diagnostics/vecnorm_obs_var_mean", float(np.mean(env.obs_rms.var)))
        if hasattr(env, "ret_rms"):
            self.logger.record("diagnostics/vecnorm_ret_var", float(np.mean(env.ret_rms.var)))

        # Plateau detection from completed episodes
        ep_rewards = [info["episode"]["r"] for info in self.locals.get("infos", []) if "episode" in info]
        if ep_rewards:
            self._rollout_ep_rewards.append(np.mean(ep_rewards))
            if len(self._rollout_ep_rewards) >= self.plateau_window:
                recent = self._rollout_ep_rewards[-self.plateau_window :]
                variation = max(recent) - min(recent)
                self.logger.record("diagnostics/reward_variation", variation)
                if variation < self.plateau_threshold:
                    print(
                        f"\n*** PLATEAU WARNING: Reward variation over last "
                        f"{self.plateau_window} rollouts is only {variation:.4f}. "
                        f"Consider adjusting learning rate or stopping. ***\n"
                    )

    def _save_diagnostics(self) -> None:
        """Persist accumulated diagnostics to an npz file in the stage dir."""
        if self._log_dir is None:
            return
        save_dict = {"timesteps": np.array(self._history_timesteps)}
        for key in self.INFO_KEYS:
            save_dict[key] = np.array(self._history[key])
        for key in self.REWARD_KEYS:
            arr = self._history_rewards[key]
            if len(arr) == len(self._history_timesteps):
                save_dict[key] = np.array(arr)
        if self._history_term_timesteps:
            save_dict["term_timesteps"] = np.array(self._history_term_timesteps)
            for reason, fracs in self._history_terminations.items():
                save_dict[f"term_{reason}"] = np.array(fracs)
        np.savez(self._log_dir / "diagnostics.npz", **save_dict)


def _eval_forward_vel(model, stage, vecnorm_path, n_episodes=25):
    """Evaluate a trained policy collecting forward velocity and success rate.

    Uses the library's create_vec_env and load_vecnorm_stats for environment
    setup, then runs evaluation episodes to collect per-episode metrics that
    SB3's EvalCallback does not capture (forward velocity, success rate).
    """
    eval_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, 1, SEED + 3000)
    if vecnorm_path and Path(vecnorm_path).exists():
        load_vecnorm_stats(vecnorm_path, eval_env)
        eval_env.training = False
        eval_env.norm_reward = False

    rewards, lengths, fwd_vels, successes = [], [], [], []
    for _ in range(n_episodes):
        obs = eval_env.reset()
        ep_reward, ep_len, ep_vels, ep_success = 0.0, 0, [], 0.0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, dones, infos = eval_env.step(action)
            ep_reward += float(reward[0])
            ep_len += 1
            if "forward_vel" in infos[0]:
                ep_vels.append(float(infos[0]["forward_vel"]))
            for k in SPECIES_CFG.success_keys:
                if infos[0].get(k):
                    ep_success = 1.0
                    break
            done = bool(dones[0])
        rewards.append(ep_reward)
        lengths.append(float(ep_len))
        fwd_vels.append(float(np.mean(ep_vels)) if ep_vels else 0.0)
        successes.append(ep_success)

    eval_env.close()
    return rewards, lengths, fwd_vels, successes


def record_stage_video(model, stage, stage_dir, vecnorm_path=None, max_steps=1000):
    """Record and save a video of the trained policy for a given stage.

    When vecnorm_path is provided, observations are normalized using the saved
    VecNormalize running statistics so the policy sees the same input distribution
    it was trained on.
    """
    if not _HAS_MEDIAPY:
        print(f"Skipping video for stage {stage} (mediapy not installed).")
        return

    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

    env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
    render_env = EnvClass(render_mode="rgb_array", **env_kwargs)

    # Load VecNormalize stats so observations match the training distribution.
    # Without this, the model receives unnormalized obs and produces bad actions.
    vec_normalize = None
    if vecnorm_path and Path(vecnorm_path).exists():
        dummy_env = DummyVecEnv([lambda: EnvClass(**env_kwargs)])
        vec_normalize = VecNormalize.load(vecnorm_path, dummy_env)
        vec_normalize.training = False
        vec_normalize.norm_reward = False

    obs, _ = render_env.reset(seed=SEED + 2000 + stage)
    frames = []
    episode_reward = 0.0

    for _ in range(max_steps):
        # Normalize observation if VecNormalize stats are available
        if vec_normalize is not None:
            obs_input = vec_normalize.normalize_obs(obs)
        else:
            obs_input = obs
        action, _ = model.predict(obs_input, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(action)
        frames.append(render_env.render())
        episode_reward += reward
        if terminated or truncated:
            break

    render_env.close()
    if vec_normalize is not None:
        vec_normalize.close()

    video_path = str(Path(stage_dir) / f"{SPECIES}_{ALGORITHM.lower()}_stage{stage}.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Stage {stage} video: reward={episode_reward:.2f} | {len(frames)} frames")
    print(f"  Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)


def train_stage(
    stage,
    timesteps,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
):
    """Train a single curriculum stage.

    Uses library functions for environment creation (``create_vec_env``),
    VecNormalize transfer (``load_vecnorm_stats``), and LR scheduling
    (``linear_schedule``).  The notebook-specific ``DiagnosticsCallback``
    is added on top of the standard SB3 callbacks.

    Returns (model, best_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    stage_start = time.time()

    config = STAGE_CONFIGS[stage]
    AlgoClass = ALGO_CLASS[ALGORITHM]

    # Algorithm hyperparameters from TOML config
    algo_key = "sac_kwargs" if ALGORITHM.lower() == "sac" else "ppo_kwargs"
    algo_kwargs = config[algo_key].copy()
    algo_kwargs["verbose"] = VERBOSE

    # Extract policy_kwargs from TOML config, falling back to default net_arch
    policy_kwargs = algo_kwargs.pop("policy_kwargs", {"net_arch": [256, 256]})

    # Handle learning_rate_end: create a linear schedule using the library helper
    lr_end = algo_kwargs.pop("learning_rate_end", None)
    if lr_end is not None:
        lr_start = algo_kwargs["learning_rate"]
        algo_kwargs["learning_rate"] = linear_schedule(lr_start, lr_end)

    # Directories - each stage gets a subdirectory within the run directory
    if run_dir is None:
        run_dir = repo_root / "logs"
    stage_dir = Path(run_dir) / f"stage{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)
    algo_kwargs["tensorboard_log"] = str(stage_dir / "tensorboard")

    print(f"{'=' * 60}")
    print(f"Stage {stage}: {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize from prior stage: {vecnorm_path}")
    print(f"{'=' * 60}")

    # Resolve stage transition parameters up front so the exact values
    # (whether from TOML or defaults) are recorded in the saved config.
    cur_kwargs = config.get("curriculum_kwargs", {})
    warmup_timesteps = cur_kwargs.get("warmup_timesteps", 100_000)
    warmup_clip_range = cur_kwargs.get("warmup_clip_range", 0.02)
    warmup_ent_coef = cur_kwargs.get("warmup_ent_coef", 0.02)
    ramp_timesteps = cur_kwargs.get("ramp_timesteps", 500_000)
    ramp_start_value = cur_kwargs.get("ramp_start_value", 0.1)
    target_fwd_weight = config["env_kwargs"].get("forward_vel_weight", 1.0)

    # Save reward weights, hyperparameters, and resolved stage transition
    # params for reproducibility.  The "run" section captures the actual
    # values used (including defaults) so a run can be replicated exactly.
    cfg_path = save_stage_config(
        stage_dir,
        stage,
        config,
        ALGORITHM,
        extra={
            "seed": SEED,
            "n_envs": N_ENVS,
            "timesteps": timesteps,
            "warmup_timesteps": warmup_timesteps,
            "warmup_clip_range": warmup_clip_range,
            "warmup_ent_coef": warmup_ent_coef,
            "ramp_timesteps": ramp_timesteps,
            "ramp_start_value": ramp_start_value,
            "forward_vel_weight": target_fwd_weight,
        },
    )
    print(f"Stage config saved to: {cfg_path}")

    # Create environments using library infrastructure
    train_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, N_ENVS, SEED)
    eval_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, 1, SEED + 1000)

    # Load VecNormalize stats from prior stage using library
    if vecnorm_path:
        load_vecnorm_stats(vecnorm_path, train_env, eval_env)

    # Create or load model
    if load_path:
        print(f"Loading model from: {load_path}")
        model = AlgoClass.load(
            load_path,
            env=train_env,
            tensorboard_log=algo_kwargs.get("tensorboard_log")
        )
        # SB3 expects schedule functions, not raw floats
        if callable(algo_kwargs["learning_rate"]):
            model.learning_rate = algo_kwargs["learning_rate"]
        else:
            model.learning_rate = get_schedule_fn(algo_kwargs["learning_rate"])
        if ALGORITHM == "PPO":
            model.ent_coef = algo_kwargs["ent_coef"]
            model.clip_range = get_schedule_fn(algo_kwargs["clip_range"])
    else:
        model = AlgoClass(
            "MlpPolicy",
            train_env,
            policy_kwargs=policy_kwargs,
            **algo_kwargs,
        )

    # Callbacks
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(model_dir),
        log_path=str(stage_dir),
        eval_freq=max(50000 // N_ENVS, 1),
        n_eval_episodes=25,
        deterministic=True,
        verbose=max(VERBOSE, 1),
    )
    checkpoint_callback = CheckpointCallback(
        save_freq=max(50000 // N_ENVS, 1),
        save_path=str(model_dir),
        name_prefix=f"stage{stage}",
        save_vecnormalize=True,
    )
    diagnostics_callback = DiagnosticsCallback(
        plateau_window=10,
        plateau_threshold=1.0,
        log_dir=str(stage_dir),
    )

    # Stage transition callbacks from library (stages 2+): warm up the value
    # function and gradually ramp the forward velocity reward to prevent
    # catastrophic forgetting of balance behaviours.
    stage_transition_cbs = []
    if stage > 1 and ALGORITHM == "PPO":
        stage_transition_cbs.append(StageWarmupCallback(
            warmup_timesteps=warmup_timesteps,
            warmup_clip_range=warmup_clip_range,
            warmup_ent_coef=warmup_ent_coef,
        ))
    if stage > 1:
        stage_transition_cbs.append(RewardRampCallback(
            attr_name="forward_vel_weight",
            start_value=ramp_start_value,
            end_value=target_fwd_weight,
            ramp_timesteps=ramp_timesteps,
        ))

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=CallbackList([eval_callback, checkpoint_callback, diagnostics_callback] + stage_transition_cbs),
        progress_bar=VERBOSE >= 1,
    )

    # Save final model and VecNormalize stats
    final_path = model_dir / f"stage{stage}_final"
    model.save(str(final_path))
    vecnorm_save_path = str(final_path) + "_vecnorm.pkl"
    train_env.save(vecnorm_save_path)
    print(f"\nFinal model saved to: {final_path}.zip")
    print(f"VecNormalize stats saved to: {vecnorm_save_path}")

    # Use best model for video recording and next-stage loading
    best_model_zip = model_dir / "best_model.zip"
    if best_model_zip.exists():
        best_path = model_dir / "best_model"
        model = AlgoClass.load(str(best_path), env=train_env)
        output_path = str(best_path)
        print(f"Loaded best model for video/next-stage: {best_path}.zip")
    else:
        output_path = str(final_path)

    # Post-training evaluation with forward velocity measurement
    episode_rewards, episode_lengths, episode_fwd_vels, episode_successes = _eval_forward_vel(
        model, stage, vecnorm_save_path, n_episodes=25,
    )
    mean_reward = float(np.mean(episode_rewards))
    std_reward = float(np.std(episode_rewards))
    mean_length = float(np.mean(episode_lengths))
    std_length = float(np.std(episode_lengths))
    mean_fwd_vel = float(np.mean(episode_fwd_vels))
    std_fwd_vel = float(np.std(episode_fwd_vels))
    sim_dt = float(eval_env.get_attr("dt")[0])
    mean_success_rate = float(np.mean(episode_successes))
    print(f"Eval: mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Eval: mean_length={mean_length:.1f} +/- {std_length:.1f} steps ({mean_length * sim_dt:.2f}s sim time)")
    print(f"Eval: mean_forward_vel={mean_fwd_vel:.2f} +/- {std_fwd_vel:.2f} m/s")
    print(f"Eval: success_rate={mean_success_rate:.0%}")

    train_env.close()
    eval_env.close()

    stage_duration = time.time() - stage_start

    # Check curriculum gating thresholds
    target_reward = cur_kwargs.get("min_avg_reward", -float("inf"))
    target_length = cur_kwargs.get("min_avg_episode_length", 0.0)
    target_fwd_vel = cur_kwargs.get("min_avg_forward_vel", 0.0)
    target_success_rate = cur_kwargs.get("min_success_rate", 0.0)

    gate_failures = []
    if mean_reward < target_reward:
        gate_failures.append(f"mean reward {mean_reward:.2f} < {target_reward:.2f}")
    if mean_length < target_length:
        gate_failures.append(f"mean episode length {mean_length:.1f} < {target_length:.1f}")
    if target_fwd_vel > 0.0 and mean_fwd_vel < target_fwd_vel:
        gate_failures.append(f"mean forward vel {mean_fwd_vel:.2f} m/s < {target_fwd_vel:.2f} m/s")
    if target_success_rate > 0.0 and mean_success_rate < target_success_rate:
        gate_failures.append(f"success rate {mean_success_rate:.0%} < {target_success_rate:.0%}")

    gate_passed = len(gate_failures) == 0
    if not gate_passed:
        print(
            f"\n*** WARNING: Stage {stage} did not meet curriculum gate: "
            + "; ".join(gate_failures)
            + ". ***"
        )

    stage_results = {
        "stage": stage,
        "name": config["name"],
        "description": config["description"],
        "timesteps": timesteps,
        "duration_seconds": stage_duration,
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "mean_episode_length": mean_length,
        "std_episode_length": std_length,
        "mean_forward_vel": mean_fwd_vel,
        "std_forward_vel": std_fwd_vel,
        "mean_success_rate": mean_success_rate,
        "sim_dt": sim_dt,
        "model_path": output_path,
        "vecnorm_path": vecnorm_save_path,
        "gate_passed": gate_passed,
        "gate_failures": gate_failures,
    }

    # Write per-stage text summary
    write_stage_summary(stage_dir, stage_results)

    return model, output_path, stage_dir, vecnorm_save_path, stage_results


def write_stage_summary(stage_dir, results_dict, species=None):
    """Write a text summary for a single completed stage to its directory."""
    if species is None:
        species = SPECIES.title()
    summary_path = Path(stage_dir) / "stage_summary.txt"
    mean_len = results_dict.get("mean_episode_length", 0)
    std_len = results_dict.get("std_episode_length", 0)
    sim_dt = results_dict.get("sim_dt", 0.01)
    avg_duration_s = mean_len * sim_dt
    mean_vel = results_dict.get("mean_forward_vel", 0.0)
    std_vel = results_dict.get("std_forward_vel", 0.0)
    lines = [
        f"Mesozoic Labs: Stage {results_dict['stage']} Summary",
        "=" * 50,
        "",
        f"Species:        {species}",
        f"Stage:          {results_dict['stage']} ({results_dict['name']})",
        f"Description:    {results_dict['description']}",
        f"Algorithm:      {ALGORITHM}",
        f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Timesteps:      {results_dict['timesteps']:,}",
        f"Duration:       {_format_duration(results_dict['duration_seconds'])}",
        f"Eval reward:    {results_dict['mean_reward']:.2f} +/- {results_dict['std_reward']:.2f}",
        f"Avg ep length:  {mean_len:.1f} +/- {std_len:.1f} steps ({avg_duration_s:.2f}s sim time)",
        f"Avg fwd vel:    {mean_vel:.2f} +/- {std_vel:.2f} m/s",
        f"Best model:     {results_dict['model_path']}.zip",
        f"VecNormalize:   {results_dict['vecnorm_path']}",
        "",
    ]
    summary_text = "\n".join(lines) + "\n"
    summary_path.write_text(summary_text)
    print(f"\nStage {results_dict['stage']} summary saved to: {summary_path}")


def _format_duration(seconds):
    """Format seconds into a human-readable string."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    elif m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


def _format_duration_hms(seconds):
    """Format seconds as H:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}"


def write_training_summary(run_dir, stage_results_list, species=None):
    """Write a training summary text file to the run directory."""
    if species is None:
        species = SPECIES.title()
    summary_path = Path(run_dir) / "training_summary.txt"
    total_duration = sum(r["duration_seconds"] for r in stage_results_list)

    lines = [
        "Mesozoic Labs Training Summary",
        "=" * 50,
        "",
        f"Species:        {species}",
        f"Algorithm:      {ALGORITHM}",
        f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Seed:           {SEED}",
        f"Quick test:     {QUICK_TEST}",
        f"Parallel envs:  {N_ENVS}",
        f"Run directory:  {run_dir}",
        "",
    ]

    for r in stage_results_list:
        mean_len = r.get("mean_episode_length", 0)
        std_len = r.get("std_episode_length", 0)
        sim_dt = r.get("sim_dt", 0.01)
        mean_vel = r.get("mean_forward_vel", 0.0)
        std_vel = r.get("std_forward_vel", 0.0)
        lines.extend(
            [
                f"Stage {r['stage']}: {r['name']}",
                f"  Description:    {r['description']}",
                f"  Timesteps:      {r['timesteps']:,}",
                f"  Duration:       {_format_duration(r['duration_seconds'])}",
                f"  Eval reward:    {r['mean_reward']:.2f} +/- {r['std_reward']:.2f}",
                f"  Avg ep length:  {mean_len:.1f} +/- {std_len:.1f} steps ({mean_len * sim_dt:.2f}s sim time)",
                f"  Avg fwd vel:    {mean_vel:.2f} +/- {std_vel:.2f} m/s",
                f"  Best model:     {r['model_path']}.zip",
                "",
            ]
        )

    lines.extend(
        [
            "-" * 50,
            f"Total training time: {_format_duration(total_duration)}",
        ]
    )

    summary_text = "\n".join(lines) + "\n"
    summary_path.write_text(summary_text)
    print(f"\nTraining summary saved to: {summary_path}")
    print(summary_text)


def save_results_json(stage_results_list, species=SPECIES):
    """Save a summary.json to results/<species>/<algorithm>/ in the repo.

    This creates a machine-readable record of the training run that can be
    used to auto-generate the README results table and website content.
    """
    import json

    algo_lower = ALGORITHM.lower()
    results_dir = repo_root / "results" / species / algo_lower
    results_dir.mkdir(parents=True, exist_ok=True)

    total_duration = sum(r["duration_seconds"] for r in stage_results_list)
    total_timesteps = sum(r["timesteps"] for r in stage_results_list)
    final_result = stage_results_list[-1]

    stages = {}
    for r in stage_results_list:
        stage_data = {
            "name": r["name"],
            "timesteps": r["timesteps"],
            "avg_reward": round(r["mean_reward"], 2),
            "std_reward": round(r["std_reward"], 2),
            "training_time_seconds": round(r["duration_seconds"], 1),
            "training_time": _format_duration_hms(r["duration_seconds"]),
        }
        if "mean_forward_vel" in r:
            stage_data["avg_forward_vel"] = round(r["mean_forward_vel"], 2)
        stages[str(r["stage"])] = stage_data

    summary = {
        "species": species,
        "algorithm": ALGORITHM,
        "hardware": "Google Colab T4 GPU",
        "seed": SEED,
        "date": datetime.now().strftime("%Y-%m-%d"),
        "stages": stages,
        "total_timesteps": total_timesteps,
        "total_training_time_seconds": round(total_duration, 1),
        "total_training_time": _format_duration_hms(total_duration),
        "final_avg_reward": round(final_result["mean_reward"], 2),
    }

    summary_path = results_dir / "summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    print(f"\nResults summary saved to: {summary_path}")
    return summary_path


print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
print("VecNormalize tracking, termination reasons, and plateau detection")
print("will log to TensorBoard.")

### Visualization Functions

In [ ]:
def plot_training_curves(stage_dirs, stage_configs, algo_name, save_path=None):
    """Plot evaluation reward, episode length, tilt angle, and forward velocity curves.

    Produces a 2x2 grid:
      - Top-left: Eval Reward
      - Top-right: Episode Length
      - Bottom-left: Tilt Angle
      - Bottom-right: Forward Velocity (Speed)

    If save_path is provided, the figure is saved as a PNG to that location.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for stage_num, stage_dir in stage_dirs:
        eval_log = Path(stage_dir) / "evaluations.npz"
        if not eval_log.exists():
            print(f"No evaluation log found for stage {stage_num}.")
            continue

        data = np.load(eval_log)
        timesteps = data["timesteps"]
        results = data["results"]
        label = f"Stage {stage_num}: {stage_configs[stage_num]['name']}"

        # Reward curve
        mean_rewards = np.mean(results, axis=1)
        std_rewards = np.std(results, axis=1)
        axes[0, 0].plot(timesteps, mean_rewards, label=label)
        axes[0, 0].fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, alpha=0.2)

        # Episode length curve (ep_lengths saved by SB3 EvalCallback)
        if "ep_lengths" in data:
            ep_lengths = data["ep_lengths"]
            mean_lengths = np.mean(ep_lengths, axis=1)
            std_lengths = np.std(ep_lengths, axis=1)
            axes[0, 1].plot(timesteps, mean_lengths, label=label)
            axes[0, 1].fill_between(timesteps, mean_lengths - std_lengths, mean_lengths + std_lengths, alpha=0.2)

        # Draw curriculum threshold lines
        cur = stage_configs[stage_num].get("curriculum_kwargs", {})
        min_reward = cur.get("min_avg_reward")
        min_length = cur.get("min_avg_episode_length")
        color = axes[0, 0].get_lines()[-1].get_color()
        if min_reward is not None:
            axes[0, 0].axhline(y=min_reward, color=color, linestyle="--", alpha=0.5)
        if min_length is not None:
            axes[0, 1].axhline(y=min_length, color=color, linestyle="--", alpha=0.5)

        # Tilt angle and forward velocity from diagnostics
        diag_log = Path(stage_dir) / "diagnostics.npz"
        if diag_log.exists():
            diag = np.load(diag_log)
            if "tilt_angle" in diag and "timesteps" in diag:
                diag_ts = diag["timesteps"]
                tilt = np.degrees(diag["tilt_angle"])
                axes[1, 0].plot(diag_ts, tilt, label=label, color=color)
            if "forward_vel" in diag and "timesteps" in diag:
                diag_ts = diag["timesteps"]
                fwd_vel = diag["forward_vel"]
                axes[1, 1].plot(diag_ts, fwd_vel, label=label, color=color)

    axes[0, 0].set_xlabel("Timesteps")
    axes[0, 0].set_ylabel("Mean Reward")
    axes[0, 0].set_title(f"{SPECIES.title()} {algo_name} - Eval Reward")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].set_xlabel("Timesteps")
    axes[0, 1].set_ylabel("Mean Episode Length (steps)")
    axes[0, 1].set_title(f"{SPECIES.title()} {algo_name} - Episode Length")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].set_xlabel("Timesteps")
    axes[1, 0].set_ylabel("Mean Tilt Angle (degrees)")
    axes[1, 0].set_title(f"{SPECIES.title()} {algo_name} - Tilt Angle")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].set_xlabel("Timesteps")
    axes[1, 1].set_ylabel("Mean Forward Velocity (m/s)")
    axes[1, 1].set_title(f"{SPECIES.title()} {algo_name} - Speed")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Training curves saved to: {save_path}")

    plt.show()



def plot_diagnostics_graphs(stage_dirs, stage_configs, algo_name, save_dir=None, _show=True):
    """Create two 2\u00d72 diagnostic figures tracking advanced training metrics.

    Figure 1 \u2013 Locomotion Health (saved as ``locomotion_health.png``):
      - [0,0] Termination Breakdown
      - [0,1] Cost of Transport
      - [1,0] Pelvis Height
      - [1,1] Reward Decomposition

    Figure 2 \u2013 Behavioral Metrics (saved as ``behavioral_metrics.png``):
      - [0,0] Gait Symmetry + Stride Frequency
      - [0,1] Heading Alignment
      - [1,0] Prey Distance
      - [1,1] Strike Success Rate
    """
    _SPECIES = SPECIES.title()
    _PELVIS_KEY = "pelvis_height"
    _PELVIS_LABEL = "Pelvis Height (m)"
    _SUCCESS_KEY = "strike_success"
    _SUCCESS_LABEL = "Strike Success Rate"
    _PREY_KEY = "prey_distance"
    _PREY_LABEL = "Prey Distance (m)"
    _HAS_HEADING = True
    _REWARD_COMPONENTS = [k for k in DiagnosticsCallback.REWARD_KEYS if k != "reward_total"]

    # -----------------------------------------------------------
    # Figure 1: Locomotion Health Metrics
    # -----------------------------------------------------------
    fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
    fig1.suptitle(f"{_SPECIES} {algo_name} \u2013 Locomotion Health", fontsize=14, fontweight="bold")
    _reward_colors = plt.cm.tab10.colors
    _all_term_data = {}  # label -> {reason: mean_fraction}

    for stage_num, stage_dir in stage_dirs:
        diag_log = Path(stage_dir) / "diagnostics.npz"
        if not diag_log.exists():
            continue
        diag = np.load(diag_log)
        label = f"Stage {stage_num}: {stage_configs[stage_num]['name']}"
        ts = diag["timesteps"] if "timesteps" in diag else None
        if ts is None or len(ts) == 0:
            continue
        color = _reward_colors[stage_num % 10]

        # [0,0] Collect termination breakdown per stage
        _term_keys = [k for k in diag.files if k.startswith("term_") and k != "term_timesteps"]
        if _term_keys:
            _all_term_data[label] = {k[5:]: float(np.mean(diag[k])) for k in _term_keys}

        # [0,1] Cost of Transport: |reward_energy| / max(forward_vel, 0.01)
        if "reward_energy" in diag and "forward_vel" in diag:
            _energy = np.abs(diag["reward_energy"])
            _fwd = np.maximum(diag["forward_vel"], 0.01)
            axes1[0, 1].plot(ts, _energy / _fwd, label=label, color=color)

        # [1,0] Pelvis Height
        if _PELVIS_KEY in diag:
            axes1[1, 0].plot(ts, diag[_PELVIS_KEY], label=label, color=color)

        # [1,1] Reward Decomposition
        for _ci, _rkey in enumerate(_REWARD_COMPONENTS):
            if _rkey in diag:
                axes1[1, 1].plot(
                    ts, diag[_rkey],
                    label=f"S{stage_num} {_rkey.replace('reward_', '')}",
                    color=_reward_colors[_ci % len(_reward_colors)],
                    linestyle=["-", "--", "-."][stage_num % 3],
                )

    # Render termination breakdown as grouped bar chart
    _ax_term = axes1[0, 0]
    if _all_term_data:
        _all_reasons = sorted({r for d in _all_term_data.values() for r in d})
        _x = np.arange(len(_all_term_data))
        _w = 0.8 / max(len(_all_reasons), 1)
        for _ri, _reason in enumerate(_all_reasons):
            _vals = [_all_term_data[_lbl].get(_reason, 0.0) for _lbl in _all_term_data]
            _offset = (_ri - len(_all_reasons) / 2 + 0.5) * _w
            _ax_term.bar(_x + _offset, _vals, _w, label=_reason)
        _ax_term.set_xticks(_x)
        _ax_term.set_xticklabels(list(_all_term_data.keys()), rotation=15, ha="right", fontsize=8)
        _ax_term.legend(fontsize=7, loc="upper right")
    else:
        _ax_term.text(0.5, 0.5, "No termination data yet", transform=_ax_term.transAxes,
                      ha="center", va="center", fontsize=12, color="gray")
    _ax_term.set_title(f"{_SPECIES} {algo_name} \u2013 Termination Breakdown")
    _ax_term.set_ylabel("Mean Fraction")
    _ax_term.grid(True, alpha=0.3)

    axes1[0, 1].set_xlabel("Timesteps")
    axes1[0, 1].set_ylabel("Cost of Transport (energy / speed)")
    axes1[0, 1].set_title(f"{_SPECIES} {algo_name} \u2013 Cost of Transport")
    axes1[0, 1].legend(fontsize=8)
    axes1[0, 1].grid(True, alpha=0.3)

    axes1[1, 0].set_xlabel("Timesteps")
    axes1[1, 0].set_ylabel(_PELVIS_LABEL)
    axes1[1, 0].set_title(f"{_SPECIES} {algo_name} \u2013 Pelvis Height")
    axes1[1, 0].legend()
    axes1[1, 0].grid(True, alpha=0.3)

    axes1[1, 1].set_xlabel("Timesteps")
    axes1[1, 1].set_ylabel("Mean Reward Component")
    axes1[1, 1].set_title(f"{_SPECIES} {algo_name} \u2013 Reward Decomposition")
    axes1[1, 1].legend(fontsize=7, loc="upper left")
    axes1[1, 1].grid(True, alpha=0.3)

    fig1.tight_layout()
    if save_dir is not None:
        _p1 = Path(save_dir) / "locomotion_health.png"
        fig1.savefig(_p1, dpi=150, bbox_inches="tight")
        print(f"Locomotion health graph saved to: {_p1}")
    if _show:
        plt.show()
    else:
        plt.close(fig1)

    # -----------------------------------------------------------
    # Figure 2: Behavioral Metrics
    # -----------------------------------------------------------
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    fig2.suptitle(f"{_SPECIES} {algo_name} \u2013 Behavioral Metrics", fontsize=14, fontweight="bold")

    for stage_num, stage_dir in stage_dirs:
        diag_log = Path(stage_dir) / "diagnostics.npz"
        if not diag_log.exists():
            continue
        diag = np.load(diag_log)
        label = f"Stage {stage_num}: {stage_configs[stage_num]['name']}"
        ts = diag["timesteps"] if "timesteps" in diag else None
        if ts is None or len(ts) == 0:
            continue
        color = plt.cm.tab10.colors[stage_num % 10]

        # [0,0] Gait Symmetry + Stride Frequency proxy
        if "l_foot_contact" in diag and "r_foot_contact" in diag:
            _l = diag["l_foot_contact"]
            _r = diag["r_foot_contact"]
            _gait_sym = 1.0 - np.abs(_l - _r) / (_l + _r + 1e-8)
            _stride_proxy = (_l + _r) / 2.0
            axes2[0, 0].plot(ts, _gait_sym, label=f"{label} \u2013 gait sym", color=color, linestyle="-")
            axes2[0, 0].plot(ts, _stride_proxy, label=f"{label} \u2013 stride freq", color=color, linestyle="--", alpha=0.7)

        # [0,1] Heading Alignment
        if _HAS_HEADING and "heading_alignment" in diag:
            axes2[0, 1].plot(ts, diag["heading_alignment"], label=label, color=color)

        # [1,0] Prey / Food Distance
        if _PREY_KEY in diag:
            axes2[1, 0].plot(ts, diag[_PREY_KEY], label=label, color=color)

        # [1,1] Strike / Bite / Food Success Rate
        if _SUCCESS_KEY in diag:
            axes2[1, 1].plot(ts, diag[_SUCCESS_KEY], label=label, color=color)

    axes2[0, 0].set_xlabel("Timesteps")
    axes2[0, 0].set_ylabel("Gait Symmetry (\u2013) / Stride Freq proxy (--)")
    axes2[0, 0].set_title(f"{_SPECIES} {algo_name} \u2013 Gait Symmetry + Stride Frequency")
    axes2[0, 0].legend(fontsize=8)
    axes2[0, 0].grid(True, alpha=0.3)

    axes2[0, 1].set_xlabel("Timesteps")
    axes2[0, 1].set_ylabel("Heading Alignment (cos \u03b8)")
    axes2[0, 1].set_title(f"{_SPECIES} {algo_name} \u2013 Heading Alignment")

    axes2[0, 1].legend()
    axes2[0, 1].grid(True, alpha=0.3)

    axes2[1, 0].set_xlabel("Timesteps")
    axes2[1, 0].set_ylabel(_PREY_LABEL)
    axes2[1, 0].set_title(f"{_SPECIES} {algo_name} \u2013 Prey Distance")
    axes2[1, 0].legend()
    axes2[1, 0].grid(True, alpha=0.3)

    axes2[1, 1].set_xlabel("Timesteps")
    axes2[1, 1].set_ylabel(_SUCCESS_LABEL)
    axes2[1, 1].set_title(f"{_SPECIES} {algo_name} \u2013 {_SUCCESS_LABEL}")
    axes2[1, 1].legend()
    axes2[1, 1].grid(True, alpha=0.3)

    fig2.tight_layout()
    if save_dir is not None:
        _p2 = Path(save_dir) / "behavioral_metrics.png"
        fig2.savefig(_p2, dpi=150, bbox_inches="tight")
        print(f"Behavioral metrics graph saved to: {_p2}")
    if _show:
        plt.show()
    else:
        plt.close(fig2)

    # Save per-stage diagnostic figures to each stage's directory
    if save_dir is not None and _show:
        for _stage_num, _stage_dir in stage_dirs:
            plot_diagnostics_graphs(
                [(_stage_num, _stage_dir)], stage_configs, algo_name,
                save_dir=_stage_dir, _show=False,
            )


## 5. Stage 1: Balance

The raptor learns to stand upright without falling. No forward velocity reward —
just a strong alive bonus.

In [ ]:
cur1 = STAGE_CONFIGS[1].get("curriculum_kwargs", {})
timesteps_1 = 50_000 if QUICK_TEST else cur1.get("timesteps", 1_000_000)

model_1, path_1, dir_1, vecnorm_1, results_1 = train_stage(stage=1, timesteps=timesteps_1, run_dir=RUN_DIR)

In [ ]:
record_stage_video(model_1, stage=1, stage_dir=dir_1, vecnorm_path=vecnorm_1)

# Save individual stage training curves to stage directory
completed_stages.append((1, dir_1))
plot_training_curves([(1, dir_1)], STAGE_CONFIGS, ALGORITHM, save_path=Path(dir_1) / "training_curves.png")

# Save master training curves with all stages completed so far
plot_training_curves(completed_stages, STAGE_CONFIGS, ALGORITHM, save_path=RUN_DIR / "training_curves.png")
plot_diagnostics_graphs(completed_stages, STAGE_CONFIGS, ALGORITHM, save_dir=RUN_DIR)

# Enforce curriculum gate after recording video and saving curves
if not results_1["gate_passed"]:
    raise RuntimeError(
        "Stage 1 failed curriculum gate: "
        + "; ".join(results_1["gate_failures"])
        + "."
    )

## 6. Stage 2: Locomotion

Starting from the Stage 1 checkpoint, the raptor learns to walk and run forward.

In [ ]:
cur2 = STAGE_CONFIGS[2].get("curriculum_kwargs", {})
timesteps_2 = 50_000 if QUICK_TEST else cur2.get("timesteps", 2_000_000)

model_2, path_2, dir_2, vecnorm_2, results_2 = train_stage(
    stage=2, timesteps=timesteps_2, load_path=path_1, run_dir=RUN_DIR, vecnorm_path=vecnorm_1
)

In [ ]:
record_stage_video(model_2, stage=2, stage_dir=dir_2, vecnorm_path=vecnorm_2)

# Save individual stage training curves to stage directory
completed_stages.append((2, dir_2))
plot_training_curves([(2, dir_2)], STAGE_CONFIGS, ALGORITHM, save_path=Path(dir_2) / "training_curves.png")

# Save master training curves with all stages completed so far
plot_training_curves(completed_stages, STAGE_CONFIGS, ALGORITHM, save_path=RUN_DIR / "training_curves.png")
plot_diagnostics_graphs(completed_stages, STAGE_CONFIGS, ALGORITHM, save_dir=RUN_DIR)

# Enforce curriculum gate after recording video and saving curves
if not results_2["gate_passed"]:
    raise RuntimeError(
        "Stage 2 failed curriculum gate: "
        + "; ".join(results_2["gate_failures"])
        + "."
    )

## 7. Stage 3: Species-Specific Behavior

The final stage enables the species-specific reward (strike/bite/food reach) and trains the full behavioral repertoire. The stage 3 label and description are configured in the species selection cell above.

In [ ]:
cur3 = STAGE_CONFIGS[3].get("curriculum_kwargs", {})
timesteps_3 = 50_000 if QUICK_TEST else cur3.get("timesteps", 3_000_000)

model_3, path_3, dir_3, vecnorm_3, results_3 = train_stage(
    stage=3, timesteps=timesteps_3, load_path=path_2, run_dir=RUN_DIR, vecnorm_path=vecnorm_2
)

In [ ]:
record_stage_video(model_3, stage=3, stage_dir=dir_3, vecnorm_path=vecnorm_3)

# Save individual stage training curves to stage directory
completed_stages.append((3, dir_3))
plot_training_curves([(3, dir_3)], STAGE_CONFIGS, ALGORITHM, save_path=Path(dir_3) / "training_curves.png")

# Save master training curves with all stages completed so far
plot_training_curves(completed_stages, STAGE_CONFIGS, ALGORITHM, save_path=RUN_DIR / "training_curves.png")
plot_diagnostics_graphs(completed_stages, STAGE_CONFIGS, ALGORITHM, save_dir=RUN_DIR)

# Enforce curriculum gate after recording video and saving curves
if not results_3["gate_passed"]:
    raise RuntimeError(
        "Stage 3 failed curriculum gate: "
        + "; ".join(results_3["gate_failures"])
        + "."
    )

## 8. Evaluate Final Policy

In [ ]:
# Evaluate the final Stage 3 policy using the library's evaluate() function,
# which provides full locomotion metrics (gait symmetry, cost of transport,
# stride frequency, velocity consistency, etc.) via LocomotionMetrics.
print(f"Evaluating final Stage 3 policy ({ALGORITHM})...")
evaluate(
    species_cfg=SPECIES_CFG,
    stage_configs=STAGE_CONFIGS,
    model_path=path_3 + ".zip",
    n_episodes=10,
    render=False,
    stage=3,
    algorithm=ALGORITHM.lower(),
)

## 9. Training Curves

In [ ]:
# Re-plot all completed stages (convenient for re-running this cell standalone)
plot_training_curves(completed_stages, STAGE_CONFIGS, ALGORITHM, save_path=RUN_DIR / "training_curves.png")
plot_diagnostics_graphs(completed_stages, STAGE_CONFIGS, ALGORITHM, save_dir=RUN_DIR)

## 10. Record All Stage Videos

Re-record videos for all three training stages. This is useful if you want to
regenerate videos without re-running training.

In [ ]:
for stage, model, stage_dir, vecnorm in [
    (1, model_1, dir_1, vecnorm_1),
    (2, model_2, dir_2, vecnorm_2),
    (3, model_3, dir_3, vecnorm_3),
]:
    print(f"\n{'=' * 40}")
    print(f"Recording Stage {stage}: {STAGE_CONFIGS[stage]['name']}")
    print(f"{'=' * 40}")
    record_stage_video(model, stage=stage, stage_dir=stage_dir, vecnorm_path=vecnorm)

## 11. Cleanup

In [ ]:
write_training_summary(RUN_DIR, [results_1, results_2, results_3])
save_results_json([results_1, results_2, results_3], species=SPECIES)

print("Training complete!")
print(f"\nAlgorithm: {ALGORITHM}")
print(f"Run directory: {RUN_DIR}")
print(f"Stage 1 model: {path_1}.zip")
print(f"Stage 2 model: {path_2}.zip")
print(f"Stage 3 model: {path_3}.zip")
print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")